# Structured Output
We can ask the model to give output in a specified schema.The structure can be used for processing in later stages


In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv  import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7ff14ed62770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7ff14ed62ce0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year movie got released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")



In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7ff14ed62770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7ff14ed62ce0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year movie got released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'r

In [7]:
## lets compare the output
response_unstruct=model.invoke("Provide me details about inception")
print(response_unstruct.content)
response_struct=model_with_structure.invoke("Provide me details about movie Inception")
print("Structured output----")
print(response_struct)

<think>
Okay, I need to provide details about Inception. Let me start by recalling what I know. Inception is a 2010 movie directed by Christopher Nolan. The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, and Ellen Page. The story revolves around Dom Cobb, a professional thief who steals secrets by infiltrating the subconscious. Instead of just stealing, he's offered a chance to have his criminal history erased by performing the inverse: planting an idea into someone's mind. That's called "inception."

I should explain the concept of shared dreaming. The characters use a device called the PASIV (Portable Automated Somnacin IntraVenous) to enter people's dreams. The team has different roles: Cobb is the Forger, Arthur is the Architect who designs the dream layers, Eames is the Forger who can impersonate people, and Ariadne is the Architect who designs the dream. 

The movie has multiple layers of dreams, each deeper than the last. There's a risk of getting stuck in a dream i

In [10]:
#Message output with parsed structure
class Movie(BaseModel):
    # ... ->tells that it is a required field
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year movie got released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movies rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Provide me details about the Movie Robo")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Robo". Let me see what I need to do here. The available tool is the Movie function, which requires the title, year, director, and rating. The user didn\'t specify the year, director, or rating, so I don\'t have that information. Wait, maybe "Robo" is the title, but there could be multiple movies with that name. I should check if there\'s a specific one they\'re referring to. However, since the user hasn\'t provided additional details, I might need to make an assumption or ask for clarification. But according to the tools provided, I need to call the Movie function with all required parameters. Since I don\'t have the necessary data, I can\'t generate the function call accurately. Maybe I should inform the user that I need more information like the year or director to fetch the correct details. Alternatively, if there\'s a well-known movie titled "Robo," I could t

In [13]:
## Nested structre

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float| None=Field(None,description="Budget in million dollars")


model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide me details about the Movie Inception")
response



MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliott Page', role='Ariadne'), Actor(name='Tom Hardy', role='Jack'), Actor(name='Ken Watanabe', role='Saito')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

In [16]:
#TypedDict -- it dont do any rutime validation

from typing_extensions import TypedDict,Annotated
# we will get a plain dictionary as output structure
class MovieDict(TypedDict):
    """ A movie with details"""
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"the year movie was released"]
    director:Annotated[str,...,"the director of the movie"]
    rating:Annotated[float,...,"The movies rating out of 10"]

model_withtypeddict=model.with_structured_output(MovieDict)
response=model_withtypeddict.invoke("Please provide me details about Avatar")
response

{'director': 'James Cameron', 'rating': 7.8, 'title': 'Avatar', 'year': 2009}

In [18]:
## Nested structre

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float| None=Field(None,description="Budget in million dollars")


model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide me details about the Movie Avatar")
response

{'budget': 237000000,
 'cast': [{'name': 'Sam Worthington', 'role': 'Jake Sully'},
  {'name': 'Zoe Saldaña', 'role': 'Neytiri'},
  {'name': 'Sigourney Weaver', 'role': 'Dr. Grace Augustine'}],
 'genres': ['Science Fiction', 'Adventure'],
 'title': 'Avatar',
 'year': '2009'}

In [20]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}